In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_inters = 1000
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [2]:
with open('wizard_of_oz.txt', 'r',encoding = 'utf-8') as f:
    text=f.read()
char = sorted(set(text))
print(char)
vocab_size = len(char)


['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
string_to_int = {ch:i for i, ch in enumerate(char)}
int_to_string = {i:ch for i, ch in enumerate(char)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)


inputs:
tensor([[ 1, 72, 65, 62, 56, 58, 57,  1],
        [ 1, 72, 69, 68, 73, 11,  3,  0],
        [60, 65, 58, 10, 26, 74, 60,  9],
        [65, 58, 72, 72,  1, 37, 54, 67]], device='cuda:0')
targets:
tensor([[72, 65, 62, 56, 58, 57,  1, 73],
        [72, 69, 68, 73, 11,  3,  0,  0],
        [65, 58, 10, 26, 74, 60,  9,  1],
        [58, 72, 72,  1, 37, 54, 67, 60]], device='cuda:0')


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [6]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_Embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_Embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1) 
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



Y
5my6oujZiSlMDQ!_HxNoFQ(sPIL9XyL9Osns5:h90a?b]zLix2V[xDS)1Li4bR7bUbYJidn:']zS2EAzlJ"EocV
Cym?hLX]4,!*7ddVHYQ4I!Tmu4SI!oMnBvX,;z9kJ!5BNXv
V:s24K"&L2acxR5nj'O?P-qWYA.GLK?hPLpAzsRr7&AzudujHw_fD6ku93yq1BS&aTuPL.sxNYQABIiZP)J,AS!2HxiU(((yYQa'" u3sVDgtGo25'O!odiGB)jqo?5&tza*9_p)o&JMGeCUnvIP*"uAYR-tsSrUrUuwQ4u?jA4ov5m?bEH6B9QbXE25_UZw.gdI iSG&KSq&SX(IF?iRKIsxx.Oc
V?P:"pO!QY.qOQ_QmKkc?,z;3wg!&A"*DG8&'CyE_D3d((:Q40
'O

8sf.q,J,7,zuS9'bOW6?L!Vx93*KCmIPjN)bOv,
h0( 'pOsRGOsPDjJizS:krhCy56&3VR0( 7uk(.GQa)6A


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_inters):
    if iter % eval_iters ==0:
        losses = estimate_loss()
        print(f'step:{iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}')

    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step:0, train loss: 5.052, val loss: 5.012
step:250, train loss: 4.982, val loss: 4.966
step:500, train loss: 4.927, val loss: 4.901
step:750, train loss: 4.852, val loss: 4.835
4.941285133361816


In [8]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


;bxh;z;Bxc2Cn:'Jb)xfVmx&n&ZX-RENCyoMO!,'_bX,yNu1(TudGqbA 6C9&pM,
9WCyNo'OmfLZq"sGaLmJMp(X gf?fLgc]Q juY7'H:9_QA_eod'm6t93Mnxa2VxGeg5oaBB8,jqoA'JEOmKVidzwAHNWxFz;zSTRSI.P1e&:'J*?P]2UsP7,'MAwr,)nW.PhIz1qZ4o.&'L4odiKbHX_17b7UAyAbHVF na)-U:Wc[_y?-1UmA-*W]4[B693)Td g):2Cy9 9pn[t;*7&NN:QToue; :;Qb)oQ7dAyoDa)Y4mxLPUdN2ILUf0dZF
))mrUakRkrIdiI!CiS'93Qc"zyNvGk,i?ry'JZk&03*3"'pX:*Iv &q(JZSc,!q&L)a.qh,;PS6B3gbU(*iHTZsVRzNW 9ePUoErsIfYZ4Xv
o2jbR;o
ResIs6m1?r&V4xg*Ae6!k8,fXS5Uf5D*QKxgP&EZCy:JM(S)t(apIk]&S2 6 
